# EnsemblePredictor Tutorial

This notebook demonstrates how to load and use the `EnsemblePredictor` class
for making PROTAC/degrader activity predictions with uncertainty quantification.

## Overview

The `EnsemblePredictor` combines predictions from multiple models (XGBoost and
MLP) trained across different cross-validation folds and feature sets. It
provides:

- **Weighted ensemble predictions** from all loaded models
- **Multi-task support** — models are grouped by their training task (e.g., Dmax, DC50, Binary)
- **Uncertainty estimates** (standard deviation, confidence intervals)
- **Individual model predictions** for further analysis
- **Automatic feature encoding** via the saved datamodules
- **Categorical dropdown values** extracted from training encoders

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from tackai.ensemble_predictor import EnsemblePredictor, EnsemblePrediction, SampleInput

## 1. Loading the Ensemble

### 1.1 Loading from a directory (uniform weights)

The simplest way to create an `EnsemblePredictor` is to point it at a
directory that contains model checkpoints (`.ckpt` for MLP, `.json` for
XGBoost) and their associated datamodule files (`_hparams.yaml` +
`_state.pt`).  All models receive equal weight.

In [ ]:
ENSEMBLE_DIR = Path("path/to/unzipped/ensemble_directory")  # <-- UPDATE THIS PATH

predictor = EnsemblePredictor.from_directory(
    model_dir=ENSEMBLE_DIR,
    task="dmax",    # 'dmax', 'dc50', or 'bin'
    device="cpu",
)

print(predictor)

## 2. Making Predictions

### 2.1 Single prediction with `SampleInput`

The `SampleInput` dataclass wraps all the possible input fields.
**SMILES**, **POI** (name or sequence), and **E3 Ligase** are required.
Optional fields (cell line, assay, treatment time) are filled with defaults
when omitted.

> **Important:** `predict()` now returns a `Dict[str, EnsemblePrediction]`
> mapping each task (e.g. `'dmax'`) to its ensemble result. When only one
> task is present the dict will have a single entry.

### 1.3 Inspecting the loaded models

You can inspect which models were loaded, their types, and weights.

In [ ]:
info = predictor.get_model_info()

print(f"Task:            {info['task']}")
print(f"Label:           {info['label_name']}")
print(f"Has datamodule:  {info['has_datamodule']}")
print(f"Total models:    {info['n_models']}")
print()

# Count by type
from collections import Counter
type_counts = Counter(info["model_types"].values())
for model_type, count in type_counts.items():
    print(f"  {model_type}: {count}")

In [ ]:
for k, v in predictor.datamodules.items():
    print(f"{k}: {v}")

for k, v in predictor.model_types.items():
    print(f"{k}: {v}")

In [ ]:
# Show individual model weights (sorted by weight, descending)
weights_df = pd.DataFrame([
    {"model": name, "type": info["model_types"][name], "weight": w}
    for name, w in info["weights"].items()
]).sort_values("weight", ascending=False).reset_index(drop=True)

weights_df

## 2. Making Predictions

### 2.1 Single prediction with `SampleInput`

The `SampleInput` dataclass wraps all the possible input fields.  Only
`smiles` is strictly required — missing fields are filled with sensible
defaults.

In [ ]:
# All models in this ensemble require POI_Sequence (not just POI_Name)
BRD4_SEQ = "MSAESGPGTRLRNLPVMGDGLETSQMSTTQAQAQPQPANAASTNPPPPETSNPNKPKRQTNQ"

sample = SampleInput(
    smiles="COc1ccc(C2=NN(c3ccc(C(=O)Nc4cccc(-c5nc6ccccn6c5C)c4)cc3)C(c3ccc(F)cc3)C2)cc1",
    poi_sequence=BRD4_SEQ,
    ligase_name="CRBN",
    cell_line="Unknown",   # ordinal-encoded → uses unknown encoding
    treatment_time=24.0,
)

# predict() returns a dict: task_name → EnsemblePrediction
# EXAMPLE: Use task_results['dmax'] for Dmax predictions
task_results = predictor.predict(sample)

# For a single-task ensemble, extract the first (only) result
task_name = list(task_results.keys())[0]
result = task_results[task_name]
print(f"Task: {task_name}")
print(result.summary())
print('')
print("Full result as dict:")
print(result.to_dict())

The `predict()` method also accepts a simple dict with the same keys as `SampleInput` for convenience.

In [ ]:
# All models in this ensemble require POI_Sequence (not just POI_Name)
BRD4_SEQ = "MSAESGPGTRLRNLPVMGDGLETSQMSTTQAQAQPQPANAASTNPPPPETSNPNKPKRQTNQ"

sample = {
    'smiles': "COc1ccc(C2=NN(c3ccc(C(=O)Nc4cccc(-c5nc6ccccn6c5C)c4)cc3)C(c3ccc(F)cc3)C2)cc1",
    'poi_sequence': BRD4_SEQ,
    'ligase_name': "CRBN",
    'cell_line': "Unknown",   # ordinal-encoded → uses unknown encoding
    'treatment_time': 24.0,
}

# predict() returns a dict: task_name → EnsemblePrediction
# EXAMPLE: Use task_results['dmax'] for Dmax predictions
task_results = predictor.predict(sample)

# For a single-task ensemble, extract the first (only) result
task_name = list(task_results.keys())[0]
result = task_results[task_name]
print(f"Task: {task_name}")
print(result.summary())

### 2.2 Accessing prediction details

The `EnsemblePrediction` object extracted from the task dict contains rich information.

In [ ]:
print(f"Weighted mean (Dmax %):  {result.weighted_mean[0]:.2f}")
print(f"Uncertainty (std):       {result.uncertainty_std[0]:.2f}")
print(f"95% CI:                  [{result.ci_lower_95[0]:.2f}, {result.ci_upper_95[0]:.2f}]")
print(f"Prediction variance:     {result.prediction_variance[0]:.4f}")
print(f"Prediction range:        {result.prediction_range[0]:.2f}")
print(f"IQR:                     {result.prediction_iqr[0]:.2f}")
print(f"Models contributing:     {len(result.model_names)}")

In [ ]:
# Show individual model predictions
individual_df = pd.DataFrame([
    {
        "model": name,
        "prediction": pred[0],
        "weight": result.weights.get(name, 0),
    }
    for name, pred in result.individual_predictions.items()
]).sort_values("weight", ascending=False).reset_index(drop=True)

individual_df

### 2.3 Minimal input example

SMILES and at least one POI identifier (name or sequence) plus E3 ligase are
required. Missing optional fields (cell line, assay, treatment time) are
filled with defaults. Some models may be skipped if they need features you
haven't provided.

In [ ]:
BTK_SEQ = "MAAVILESIFLKRSQQKKKTSPLNFKKRLFLLTVHSGLSGSFVHFAHGDGPMWPGCQELPHLDQF"

minimal_sample = SampleInput(
    smiles="CC(C)(C)OC(=O)N1CCC(NCc2cccc(-c3ccc4[nH]c(N)nc4c3)c2)CC1",
    poi_sequence=BTK_SEQ,
    ligase_name="VHL",
)

task_results_min = predictor.predict(minimal_sample)
for t, r in task_results_min.items():
    print(f"[{t}] {r.weighted_mean[0]:.2f} ± {r.uncertainty_std[0]:.2f}")

## 3. Batch Predictions

### 3.1 Predicting from a list of `SampleInput`s

In [ ]:
SMARCA2_SEQ = "MDSYQPFKDLETKDFEYYQQLHRQKMESEQGFPSLRGAQFYHQSRLVQKKQQKIADAVEQQD"

samples = [
    SampleInput(
        smiles="COc1ccc(C2=NN(c3ccc(C(=O)Nc4cccc(-c5nc6ccccn6c5C)c4)cc3)C(c3ccc(F)cc3)C2)cc1",
        poi_sequence=BRD4_SEQ, ligase_name="CRBN", treatment_time=24.0,
    ),
    SampleInput(
        smiles="CC(C)(C)OC(=O)N1CCC(NCc2cccc(-c3ccc4[nH]c(N)nc4c3)c2)CC1",
        poi_sequence=BTK_SEQ, ligase_name="VHL", treatment_time=6.0,
    ),
    SampleInput(
        smiles="Cc1ncnc2c1cnn2-c1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1",
        poi_sequence=SMARCA2_SEQ, ligase_name="VHL", treatment_time=18.0,
    ),
]

# predict_batch returns List[Dict[str, EnsemblePrediction]]
batch_results = predictor.predict_batch(samples)

for i, task_dict in enumerate(batch_results):
    if task_dict is not None:
        for t, r in task_dict.items():
            print(f"Sample {i} [{t}]: {r.weighted_mean[0]:.2f} ± {r.uncertainty_std[0]:.2f}")
    else:
        print(f"Sample {i}: prediction failed")

### 3.2 Predicting from a DataFrame

Use `predict_dataframe` for a convenient columnar interface.

In [ ]:
df = pd.DataFrame({
    "SMILES": [
        "COc1ccc(C2=NN(c3ccc(C(=O)Nc4cccc(-c5nc6ccccn6c5C)c4)cc3)C(c3ccc(F)cc3)C2)cc1",
        "CC(C)(C)OC(=O)N1CCC(NCc2cccc(-c3ccc4[nH]c(N)nc4c3)c2)CC1",
        "Cc1ncnc2c1cnn2-c1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1",
    ],
    "POI_Sequence": [BRD4_SEQ, BTK_SEQ, SMARCA2_SEQ],
    "Ligase_Name": ["CRBN", "VHL", "VHL"],
    "Cell_Line_ID": ["Unknown", "Unknown", "Unknown"],
    "Assay_Time": [24.0, 6.0, 18.0],
})

result_df = predictor.predict_dataframe(
    df,
    smiles_col="SMILES",
    poi_col="POI_Sequence",
    ligase_col="Ligase_Name",
    cell_line_col="Cell_Line_ID",
    treatment_time_col="Assay_Time",
)

# Columns now include task-specific predictions if multi-task
result_df.filter(regex="SMILES|prediction|uncertainty|ci_")

## 4. Visualising Predictions

Below we show how to plot the individual model predictions and the
ensemble uncertainty.

In [ ]:
import matplotlib.pyplot as plt

# Re-predict a sample and extract the first task's result
task_results_viz = predictor.predict(samples[0])
result = next(iter(task_results_viz.values()))

# Collect per-model predictions
model_names = list(result.individual_predictions.keys())
preds = np.array([result.individual_predictions[n][0] for n in model_names])
weights = np.array([result.weights[n] for n in model_names])

# Sort by prediction value
order = np.argsort(preds)
model_names = [model_names[i] for i in order]
preds = preds[order]
weights = weights[order]

# Shorten long model names for display
short_names = [n.replace("model=", "")[:60] for n in model_names]

fig, ax = plt.subplots(figsize=(10, max(4, len(model_names) * 0.35)))
colors = plt.cm.Blues(weights / weights.max() * 0.7 + 0.3)
ax.barh(range(len(preds)), preds, color=colors, edgecolor="navy", alpha=0.8)
ax.axvline(result.weighted_mean[0], color="red", ls="--", lw=2,
           label=f"Ensemble mean: {result.weighted_mean[0]:.1f}")
ax.axvspan(result.ci_lower_95[0], result.ci_upper_95[0],
           alpha=0.12, color="red", label="95% CI")
ax.set_yticks(range(len(short_names)))
ax.set_yticklabels(short_names, fontsize=7)
ax.set_xlabel("Dmax (%)")
ax.set_title("Individual model predictions")
ax.legend(loc="lower right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Serialisation & Export

Prediction results can be serialised to a dictionary (for JSON export) or
inspected via the `.summary()` convenience method.

In [ ]:
import json

result_dict = result.to_dict()

# Pretty-print a subset
subset = {k: result_dict[k] for k in [
    "weighted_mean", "ci_lower_95", "ci_upper_95",
    "n_models", "task", "label_name",
]}
print(json.dumps(subset, indent=2))

## 6. Updating Weights

You can update the ensemble weights at any time without reloading the
models.  Weights are automatically re-normalised to sum to 1.

In [ ]:
# Give equal weight to all models
equal_weights = {name: 1.0 for name in predictor.models}
predictor.update_weights(equal_weights)

res_eq = next(iter(predictor.predict(samples[0]).values()))
print(f"Equal weights -> {res_eq.weighted_mean[0]:.2f} ± {res_eq.uncertainty_std[0]:.2f}")

# Give all weight to XGBoost models only
xgb_only = {
    name: 1.0 if predictor.model_types[name] == "xgboost" else 0.0
    for name in predictor.models
}
# Filter out zero-weight entries before updating
xgb_only = {k: v for k, v in xgb_only.items() if v > 0}
predictor.update_weights(xgb_only)

res_xgb = next(iter(predictor.predict(samples[0]).values()))
print(f"XGBoost only  -> {res_xgb.weighted_mean[0]:.2f} ± {res_xgb.uncertainty_std[0]:.2f}")

# Restore equal weights
predictor.update_weights(equal_weights)

## 7. Required Inputs per Model

Different models may require different input features (e.g., some need
POI sequence, some need cell descriptions). You can inspect what each
model needs.

In [ ]:
required = predictor.get_required_inputs()

# Show a compact summary: which input columns are needed by any model
all_cols = set()
for cols in required.values():
    all_cols.update(cols)

print("Columns required by at least one model:")
for col in sorted(all_cols):
    n_models = sum(1 for cols in required.values() if col in cols)
    print(f"  {col:30s}  ({n_models}/{len(required)} models)")

## 8. Multi-Task Support & Categorical Choices

### 8.1 Available tasks and model-task mapping

The predictor automatically infers each model's task from its filename or
label names. You can query the available tasks and the per-model assignment.

In [ ]:
print("Available tasks:", predictor.available_tasks)
print()

# Show which task each model is assigned to
from collections import Counter
task_counts = Counter(predictor.model_tasks.values())
for task, count in task_counts.items():
    print(f"  {task}: {count} models")

### 8.2 Categorical choices from training data

For ordinal-encoded features (Ligase, Cell Line, Assay), you can retrieve
the categories that the model's encoder saw during training. The Gradio app
uses these to populate dropdown menus.

In [ ]:
cat_choices = predictor.get_categorical_choices()

for col_name, values in cat_choices.items():
    print(f"{col_name} ({len(values)} unique values):")
    for v in values[:8]:
        print(f"  - {v}")
    if len(values) > 8:
        print(f"  ... and {len(values) - 8} more")